# Master Script — PREVISÃO DE SÉRIES TEMPORAIS (scikit-learn)

**Guia de consulta.** Não foi feito para rodar de ponta a ponta: é um repositório de
sintaxe, pipelines e grades de hiperparâmetros para copiar quando um projeto de
previsão temporal começa.

### O que muda em relação à regressão tabular comum
1. **Os dados NÃO são independentes (quebra de I.I.D.):** há autocorrelação.
   Embaralhar as linhas (`shuffle=True` ou `KFold` aleatório) vaza o futuro para o
   passado e invalida o projeto.
2. **Engenharia de variáveis temporal obrigatória:** séries brutas precisam ser
   transformadas em matriz supervisionada $(X, y)$ com lags ($y_{t-1}, y_{t-7}$),
   estatísticas móveis seguras (`shift(1).rolling()`) e componentes de calendário.
3. **Árvores NÃO extrapolam tendências:** modelos baseados em árvores (Random Forest,
   LightGBM, XGBoost, CatBoost) nunca prevêem valores fora do intervalo visto no treino.
   Se a série cresce com tendência secular, você precisa de modelos lineares,
   diferenciação ou modelos híbridos.
4. **Baselines temporais canônicos:** prever a média é ingênuo demais. Qualquer modelo
   precisa superar a **Persistência** ($\hat{y}_t = y_{t-1}$) e a **Sazonalidade Ingênua**
   ($\hat{y}_t = y_{t-7}$).

### Fluxo do documento
| # | Etapa | O que tem aqui |
|---|-------|----------------|
| 0 | Setup | Imports canônicos, `RANDOM_STATE`, config global anti-deadlock |
| 1 | Data Prep | Série sintética realista, decomposição, ACF e engenharia de lags sem vazamento |
| 2 | Split | Corte cronológico estrito 80/20, `TimeSeriesSplit` (Expanding/Rolling) e `gap` |
| 3 | Pipelines | Fábrica de pré-processamento (`scaled`, `tree`, `native`, `spline`), 22 famílias e modelo híbrido |
| 4 | Torneio | Validação cruzada temporal com `TimeSeriesSplit`, RMSE/MAE/MAPE/WAPE e direção |
| 5 | Otimização | `RandomizedSearchCV` com folds temporais, grades completas de hiperparâmetros |
| 6 | Prova Final | Backtesting no teste, gráfico temporal contínuo, análise de ruído branco (ACF) e projeção futura |
| — | Cheatsheet | Tabela de decisão rápida, sintomas/correções e os 6 erros fatais de séries temporais |


---
## 0. Setup e imports


In [ ]:
# --- Núcleo ---------------------------------------------------------------
import numpy as np
import pandas as pd

# --- Pré-processamento ----------------------------------------------------
from sklearn.compose import (
    ColumnTransformer, make_column_selector, TransformedTargetRegressor
)
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler, RobustScaler, MinMaxScaler,
    PowerTransformer, FunctionTransformer,
    OneHotEncoder, OrdinalEncoder,
    PolynomialFeatures, SplineTransformer,
)
from sklearn.feature_selection import SelectKBest, f_regression

# --- Split e validação temporal -------------------------------------------
from sklearn.model_selection import (
    train_test_split,
    TimeSeriesSplit,
    cross_validate, cross_val_predict, learning_curve,
    GridSearchCV, RandomizedSearchCV,
)

# --- Modelos: famílias matemáticas ----------------------------------------
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import (
    LinearRegression,                                  # OLS: extrapola tendência
    Ridge, Lasso, ElasticNet,                          # regularizados L2 / L1 / mistura
    BayesianRidge,                                     # dá incerteza da previsão
    HuberRegressor, QuantileRegressor,                 # robustos a choques e quantílicos
)
from sklearn.svm import LinearSVR, SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor, ExtraTreesRegressor,
    GradientBoostingRegressor, HistGradientBoostingRegressor, AdaBoostRegressor,
    VotingRegressor, StackingRegressor,
)
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# --- Métricas -------------------------------------------------------------
from sklearn.metrics import (
    mean_squared_error, root_mean_squared_error, mean_absolute_error,
    mean_absolute_percentage_error, median_absolute_error,
    r2_score, explained_variance_score, max_error,
    make_scorer, PredictionErrorDisplay,
)
from sklearn.inspection import permutation_importance

import matplotlib.pyplot as plt
import joblib
import os

# --- Config global --------------------------------------------------------
RANDOM_STATE = 42

# Paralelismo em DOIS níveis, de propósito. Cada worker do joblib abre um pool
# OpenMP do tamanho da máquina: -1 no modelo E na validação cruzada gera n_núcleos²
# threads disputando n_núcleos. Com LightGBM isso gera travamento total.
# Paralelize num nível só — o externo rende mais, cobrindo todos os modelos.
N_JOBS       = -1     # cross_validate, RandomizedSearchCV, permutation_importance
N_JOBS_MODEL = 1      # dentro do estimador (RF, ExtraTrees, XGBoost, LightGBM, CatBoost)

# Modo rápido: usado pelos testes automatizados para rodar em poucos minutos.
MODO_RAPIDO    = os.getenv("MASTER_SCRIPT_FAST") == "1"
N_LINHAS       = 365 if MODO_RAPIDO else 1_095   # 1 ano rápido vs 3 anos completos
N_ITER_BUSCA   = 5   if MODO_RAPIDO else 30      # n_iter do RandomizedSearchCV
N_REPEATS_PERM = 3   if MODO_RAPIDO else 10      # n_repeats da importância por permutação

pd.set_option("display.max_columns", 100)
plt.rcParams["figure.dpi"] = 100


In [ ]:
# --- Helper visual: inspeção gráfica dos folds de TimeSeriesSplit --------
def plot_cv_temporal(cv, X, title="Divisão dos Folds Temporais"):
    """Mostra visualmente como o TimeSeriesSplit caminha pelo tempo."""
    n_splits = cv.get_n_splits(X)
    fig, ax = plt.subplots(figsize=(10, n_splits * 0.7 + 1))
    
    indices = np.arange(len(X))
    for fold, (train_idx, val_idx) in enumerate(cv.split(X)):
        ax.scatter(train_idx, [fold] * len(train_idx), c="#4C72B0", marker="_", lw=6, label="Treino" if fold == 0 else "")
        ax.scatter(val_idx, [fold] * len(val_idx), c="#DD8452", marker="_", lw=6, label="Validação" if fold == 0 else "")
    
    ax.set_yticks(range(n_splits))
    ax.set_yticklabels([f"Fold {i+1}" for i in range(n_splits)])
    ax.set_xlabel("Índice Temporal (Linhas cronológicas)")
    ax.set_title(title)
    ax.legend(loc="upper left")
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()


---
## 1. Carregamento e Engenharia Temporal (Data Prep)

Diferente de dados tabulares estáticos, a série temporal bruta precisa ser convertida
em uma matriz supervisionada $(X, y)$.

Componentes fundamentais construídos nesta etapa:
1. **Lags de $y$:** $y_{t-1}, y_{t-2}, y_{t-7}, y_{t-14}$ — o passado imediato e sazonal.
2. **Janelas móveis seguras (`shift(1).rolling()`):** média móvel de 7 e 14 dias e volatilidade.
   > [!IMPORTANT]
   > O uso de `.shift(1)` antes do `.rolling()` é **obrigatório**. Sem ele, a média da janela
   > inclui o próprio $y_t$ que queremos prever — um vazamento catastrófico.
3. **Calendário e Ciclos:** dia da semana, mês, fim de semana e suas versões cíclicas ($\sin, \cos$).
4. **Tendência Linear:** índice $t$ que permite a modelos lineares extrapolar crescimento.


In [ ]:
# ==== 1.1 Leitura ==========================================================
# df = pd.read_csv("demanda.csv", parse_dates=["data"])
# df = pd.read_parquet("demanda.parquet")
# df = pd.read_sql("SELECT data, valor FROM serie_temporal", con=engine, parse_dates=["data"])

# ---- Série Temporal Fictícia (com sinal realista documentado) ------------
n = N_LINHAS
rng = np.random.default_rng(RANDOM_STATE)
datas = pd.date_range("2022-01-01", periods=n, freq="D")

# Componentes reais da série temporal:
# a) Tendência linear de crescimento
t = np.arange(n)
tendencia = 120.0 + 0.15 * t

# b) Sazonalidade anual (ciclo senoidal suave de 365.25 dias)
sazonalidade_anual = 20.0 * np.sin(2 * np.pi * t / 365.25)

# c) Sazonalidade semanal (efeito de finais de semana e segundas)
efeito_semanal = np.array([4.0, 1.0, 0.0, 2.0, 8.0, 22.0, 16.0])[datas.dayofweek]

# d) Variável exógena: campanhas promocionais (15% dos dias)
promocao = rng.binomial(1, 0.15, n).astype(float)
efeito_promocao = promocao * rng.uniform(15.0, 30.0, n)

# e) Choques pontuais e feriados (3% dos dias)
choques = np.zeros(n)
idx_choques = rng.choice(n, size=max(2, int(n * 0.03)), replace=False)
choques[idx_choques] = rng.uniform(30.0, 60.0, len(idx_choques))

# f) Autocorrelação de curto prazo AR(1) + ruído heterocedástico
ruido_ar = np.zeros(n)
ruido_branco = rng.normal(0, 4.0, n)
for i in range(1, n):
    ruido_ar[i] = 0.50 * ruido_ar[i - 1] + ruido_branco[i]

# Demanda total observada
demanda = tendencia + sazonalidade_anual + efeito_semanal + efeito_promocao + choques + ruido_ar
demanda = np.maximum(demanda, 10.0)

df_raw = pd.DataFrame({
    "data": datas,
    "demanda": demanda,
    "promocao": promocao,
    "id_registro": np.arange(n),
})

df_raw.head()


In [ ]:
# ==== 1.2 Diagnóstico + análise do ALVO ===================================
print(f"Dimensões brutas: {df_raw.shape}")
print(f"Período: {df_raw['data'].min().strftime('%Y-%m-%d')} até {df_raw['data'].max().strftime('%Y-%m-%d')}")

# Visualização da Série Histórica
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=False)

# 1. Série completa com média móvel de 30 dias
axes[0].plot(df_raw["data"], df_raw["demanda"], color="#4C72B0", alpha=0.6, label="Demanda Diária")
axes[0].plot(df_raw["data"], df_raw["demanda"].rolling(30).mean(), color="#C44E52", lw=2, label="Média Móvel (30 dias)")
axes[0].set_title("Demanda Diária ao Longo do Tempo (Tendência + Sazonalidade)")
axes[0].set_ylabel("Unidades")
axes[0].legend(loc="upper left")
axes[0].grid(alpha=0.3)

# 2. Sazonalidade Semanal (Boxplot por dia da semana)
dias_nomes = ["Seg", "Ter", "Qua", "Qui", "Sex", "Sáb", "Dom"]
df_raw["dia_semana_nome"] = df_raw["data"].dt.dayofweek.map(lambda x: dias_nomes[x])
df_raw.boxplot(column="demanda", by="dia_semana_nome", ax=axes[1], grid=True)
axes[1].set_title("Padrão Semanal: Variação por Dia da Semana")
axes[1].set_xlabel("")
axes[1].set_ylabel("Demanda")
fig.suptitle("")  # limpa título automático do pandas

# 3. Autocorrelograma (ACF) empírico
def plot_acf(serie, max_lags=28, ax=None):
    if ax is None:
        ax = plt.gca()
    autocorrs = [serie.autocorr(lag=l) for l in range(1, max_lags + 1)]
    lags = np.arange(1, max_lags + 1)
    ax.stem(lags, autocorrs, linefmt="#4C72B0", markerfmt="o", basefmt="k-")
    limite = 1.96 / np.sqrt(len(serie))
    ax.axhline(limite, color="r", ls="--", alpha=0.7, label=r"Intervalo 95% ($\pm 1.96/\sqrt{N}$)")
    ax.axhline(-limite, color="r", ls="--", alpha=0.7)
    ax.set_title("Função de Autocorrelação (ACF) — Picos em 7 e 14 dias indicam sazonalidade")
    ax.set_xlabel("Defasagem (Lags em dias)")
    ax.set_ylabel("Autocorrelação")
    ax.set_xticks(lags)
    ax.legend(loc="upper right")
    ax.grid(alpha=0.3)

plot_acf(df_raw["demanda"], max_lags=21, ax=axes[2])
plt.tight_layout()
plt.show()


In [ ]:
# ==== 1.3 Limpeza padrão e Engenharia de Variáveis Temporais =============
df = df_raw.copy()

# a) Remoção de colunas que não carregam valor de previsão
df = df.drop(columns=["id_registro", "dia_semana_nome"], errors="ignore")

# b) Ordenação cronológica estrita — PREMISSA FUNDAMENTAL
df = df.sort_values("data").reset_index(drop=True)

# c) Passo temporal linear (Trend Step) — essencial para modelos lineares
df["tempo_linear"] = np.arange(len(df))

# d) Lags do Alvo (Defasagens autorregressivas)
#    lag_1: ontem | lag_2: anteontem | lag_7: mesmo dia da semana anterior | lag_14: duas semanas atrás
df["lag_1"]  = df["demanda"].shift(1)
df["lag_2"]  = df["demanda"].shift(2)
df["lag_3"]  = df["demanda"].shift(3)
df["lag_7"]  = df["demanda"].shift(7)
df["lag_14"] = df["demanda"].shift(14)

# e) Janelas Móveis SEM VAZAMENTO (regra de ouro: shift(1) ANTES do rolling)
#    Se usássemos df['demanda'].rolling(7).mean() diretamente, o valor de hoje
#    estaria na média, vazando o próprio alvo para dentro das variáveis!
df["roll_mean_7"]  = df["demanda"].shift(1).rolling(window=7).mean()
df["roll_std_7"]   = df["demanda"].shift(1).rolling(window=7).std()
df["roll_mean_14"] = df["demanda"].shift(1).rolling(window=14).mean()

# f) Componentes de Calendário
df["dia_semana"]       = df["data"].dt.dayofweek
df["mes"]              = df["data"].dt.month
df["dia_mes"]          = df["data"].dt.day
df["eh_fim_de_semana"] = (df["dia_semana"] >= 5).astype(int)

# g) Codificação Cíclica (Seno / Cosseno) para manter a continuidade temporal
#    Evita que o domingo (6) pareça o extremo oposto da segunda-feira (0)
df["sin_dia_semana"] = np.sin(2 * np.pi * df["dia_semana"] / 7.0)
df["cos_dia_semana"] = np.cos(2 * np.pi * df["dia_semana"] / 7.0)
df["sin_mes"]        = np.sin(2 * np.pi * df["mes"] / 12.0)
df["cos_mes"]        = np.cos(2 * np.pi * df["mes"] / 12.0)

# h) Remoção das linhas iniciais que ficaram com NaN devido aos maiores shifts (14 dias)
n_antes = len(df)
df = df.dropna().reset_index(drop=True)
print(f"Linhas descartadas devido aos lags iniciais: {n_antes - len(df)} (sobraram {len(df)} linhas)")

TARGET = "demanda"
X = df.drop(columns=[TARGET, "data"])
y = df[TARGET]
datas_index = df["data"]

print(f"Matriz X: {X.shape} | Alvo y: {y.shape}")
X.head()


---
## 2. Separação dos Dados (Split Temporal)

Em séries temporais, **nunca embaralhe os dados**.
- O treino é sempre o passado mais antigo.
- O teste é o futuro recente (a "prova final" de inferência em dados não vistos).

| Estratégia | Como funciona | Quando usar |
|---|---|---|
| **Corte Fixo (80/20)** | Treino = primeiras 80% datas, Teste = últimas 20% | Split final da prova de inferência |
| **`TimeSeriesSplit`** | Janela expansiva (Expanding Window): o fold $k+1$ herda todo o passado do fold $k$ | Validação cruzada padrão de séries temporais |
| **Rolling Window** | Janela de tamanho fixo deslizando no tempo | Quando há quebras estruturais ou *concept drift* |
| **Purged / Gap** | Espaço de segurança entre fim do treino e início da validação | Evita vazamento via janelas móveis longas |


In [ ]:
# ==== 2.1 Split padrão: 80 / 20 ===========================================
# shuffle=False É CRÍTICO. Sem ele, o scikit-learn embaralha os dias e vaza o futuro.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    shuffle=False,
)

datas_train = datas_index.iloc[:len(X_train)]
datas_test  = datas_index.iloc[len(X_train):]

print(f"Treino: {X_train.shape} | De {datas_train.min().strftime('%Y-%m-%d')} até {datas_train.max().strftime('%Y-%m-%d')}")
print(f"Teste:  {X_test.shape}  | De {datas_test.min().strftime('%Y-%m-%d')} até {datas_test.max().strftime('%Y-%m-%d')}")
print(f"Média do alvo — treino: {y_train.mean():.2f} | teste: {y_test.mean():.2f}")


In [ ]:
# ==== 2.2 Split temporal em múltiplos folds (TimeSeriesSplit) =============
# gap=7: deixa 7 dias de espaço entre treino e validação para garantir que nenhuma
# estatística móvel calculada com o final do treino contamine o início do teste do fold.
CV = TimeSeriesSplit(n_splits=5, gap=7)

plot_cv_temporal(CV, X_train, title="TimeSeriesSplit com Janela Expansiva e Gap Anti-Vazamento")


In [ ]:
# ==== 2.3 Variantes de split ==============================================
# Janela deslizante (Rolling Window): mantém tamanho fixo de treino (ex: 180 dias)
# Útil quando regimes econômicos antigos deixam de ser representativos.
class RollingWindowSplit:
    """Gerador de índices com janela de treino de tamanho fixo deslizante."""
    def __init__(self, train_size=180, test_size=30, step=30):
        self.train_size = train_size
        self.test_size  = test_size
        self.step       = step

    def split(self, X, y=None, groups=None):
        n = len(X)
        for start in range(0, n - self.train_size - self.test_size + 1, self.step):
            train_idx = np.arange(start, start + self.train_size)
            test_idx  = np.arange(start + self.train_size, start + self.train_size + self.test_size)
            yield train_idx, test_idx

    def get_n_splits(self, X, y=None, groups=None):
        n = len(X)
        return max(0, (n - self.train_size - self.test_size) // self.step + 1)

cv_rolling = RollingWindowSplit(train_size=150, test_size=30, step=30)
print(f"Rolling Window gerou {cv_rolling.get_n_splits(X_train)} janelas deslizantes.")


---
## 3. Construção dos Pipelines

Quatro receitas de pré-processamento adaptadas às características de séries temporais:

| Receita | O que faz | Modelos indicados |
|---|---|---|
| `"scaled"` | `StandardScaler` em lags e móveis + `OneHotEncoder` em calendário | Lineares (Ridge, Lasso), SVR, KNN, MLP |
| `"tree"` | Sem escala + `OrdinalEncoder` em calendário | Árvores, Random Forest, ExtraTrees, GradientBoosting |
| `"native"` | Sem escala + preservação de variáveis nativas | LightGBM, XGBoost, CatBoost |
| `"spline"` | `SplineTransformer` em tempo e calendário + escala | Regressão Linear com curvas periódicas suaves |

> [!WARNING]
> **Árvores não extrapolam.** Se a série possui tendência de alta, um modelo de árvore
> puro nunca preverá um valor superior ao teto histórico do treino. Modelos lineares
> extrapolam a inclinação perfeitamente. Por isso, avaliamos ambos e incluímos
> a receita de modelo híbrido em 3.5.


In [ ]:
# ==== 3.1 Seleção de colunas ==============================================
LAG_COLS    = ["lag_1", "lag_2", "lag_3", "lag_7", "lag_14"]
ROLL_COLS   = ["roll_mean_7", "roll_std_7", "roll_mean_14"]
CYCLIC_COLS = ["sin_dia_semana", "cos_dia_semana", "sin_mes", "cos_mes"]
TREND_COLS  = ["tempo_linear"]
EXOG_COLS   = ["promocao", "eh_fim_de_semana"]
CAT_COLS    = ["dia_semana", "mes", "dia_mes"]

NUM_FEATURES = LAG_COLS + ROLL_COLS + CYCLIC_COLS + TREND_COLS + EXOG_COLS
CAT_FEATURES = CAT_COLS

# Como dias da semana (0..6), meses (1..12) e dias do mês (1..31) têm domínios finitos
# e conhecidos a priori, fixamos as categorias para evitar avisos em folds com poucos meses.
CATEGORIAS_CALENDARIO = [list(range(7)), list(range(1, 13)), list(range(1, 32))]

print(f"Numéricas ({len(NUM_FEATURES)}): {NUM_FEATURES}")
print(f"Categóricas de Calendário ({len(CAT_FEATURES)}): {CAT_FEATURES}")


In [ ]:
# ==== 3.2 Fábrica de pré-processadores ====================================
def build_preprocessor(kind="scaled", scaler=None, num=NUM_FEATURES, cat=CAT_FEATURES):
    """Cria ColumnTransformers modulares para séries temporais."""
    if kind == "scaled":
        num_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", scaler if scaler is not None else StandardScaler()),
        ])
        cat_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(categories=CATEGORIAS_CALENDARIO, drop="first", sparse_output=False)),
        ])
        return ColumnTransformer([
            ("num", num_pipe, num),
            ("cat", cat_pipe, cat),
        ])

    elif kind == "tree":
        num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median"))])
        cat_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
        ])
        return ColumnTransformer([
            ("num", num_pipe, num),
            ("cat", cat_pipe, cat),
        ])

    elif kind == "native":
        # Pass-through simples para LightGBM / XGBoost / CatBoost
        num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median"))])
        cat_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
        ])
        return ColumnTransformer([
            ("num", num_pipe, num),
            ("cat", cat_pipe, cat),
        ])

    elif kind == "spline":
        # Aplica splines periódicos suaves no tempo e componentes de calendário
        spline_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("spline", SplineTransformer(n_knots=5, degree=3, extrapolation="continue")),
        ])
        other_num = [c for c in num if c not in TREND_COLS]
        num_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ])
        return ColumnTransformer([
            ("spline", spline_pipe, TREND_COLS),
            ("num", num_pipe, other_num),
        ])

    raise ValueError(f"kind desconhecido: {kind!r}")


In [ ]:
# ==== 3.3 Transformação do ALVO ===========================================
def com_log_no_alvo(pipe):
    """Embrulha o pipeline com log1p no fit e expm1 no predict."""
    return TransformedTargetRegressor(
        regressor=pipe,
        func=np.log1p,
        inverse_func=np.expm1,
    )

# Em séries temporais com crescimento exponencial ou volatilidade multiplicativa,
# modelar o log da demanda estabiliza a variância dos resíduos (homocedasticidade).


In [ ]:
# ==== 3.4 Um Pipeline por família matemática ==============================

# Estimador de persistência temporal para servir de baseline obrigatório
class LagPersistenceRegressor(BaseEstimator, RegressorMixin):
    """Prevê diretamente uma coluna de lag específica (ex: lag_1 para persistência imediata)."""
    def __init__(self, col_idx=0):
        self.col_idx = col_idx

    def fit(self, X, y=None):
        self.is_fitted_ = True
        return self

    def predict(self, X):
        X_arr = np.asarray(X)
        idx = min(self.col_idx, X_arr.shape[1] - 1)
        return X_arr[:, idx]


# 22 Famílias matemáticas adaptadas para Séries Temporais
PIPELINES = {

    # ---------------- Baselines Canônicos ---------------------------------
    # Baseline 1: Média global histórica (ingênuo tabular)
    "Dummy": Pipeline([
        ("prep", build_preprocessor("tree")),
        ("model", DummyRegressor(strategy="median")),
    ]),

    # Baseline 2: Persistência / Naive (o amanhã será igual ao hoje: lag_1)
    "Persistence": Pipeline([
        ("prep", build_preprocessor("tree")),
        ("model", LagPersistenceRegressor(col_idx=0)),  # índice 0 é lag_1
    ]),

    # Baseline 3: Sazonal Ingênuo (o amanhã será igual ao mesmo dia da semana anterior: lag_7)
    "SeasonalNaive": Pipeline([
        ("prep", build_preprocessor("tree")),
        ("model", LagPersistenceRegressor(col_idx=3)),  # índice 3 é lag_7
    ]),

    # ---------------- Lineares Regularizados (Extrapolam Tendência) -------
    "LinearRegression": Pipeline([
        ("prep", build_preprocessor("scaled")),
        ("model", LinearRegression()),
    ]),

    "Ridge": Pipeline([
        ("prep", build_preprocessor("scaled")),
        ("model", Ridge(random_state=RANDOM_STATE)),
    ]),

    "Lasso": Pipeline([
        ("prep", build_preprocessor("scaled")),
        ("model", Lasso(random_state=RANDOM_STATE, max_iter=5000)),
    ]),

    "ElasticNet": Pipeline([
        ("prep", build_preprocessor("scaled")),
        ("model", ElasticNet(random_state=RANDOM_STATE, max_iter=5000)),
    ]),

    "BayesianRidge": Pipeline([
        ("prep", build_preprocessor("scaled")),
        ("model", BayesianRidge()),
    ]),

    # ---------------- Robustos e Quantílicos (Choques e Intervalos) -------
    "HuberRegressor": Pipeline([
        ("prep", build_preprocessor("scaled")),
        ("model", HuberRegressor(max_iter=1000)),
    ]),

    "QuantileRegressor": Pipeline([
        ("prep", build_preprocessor("scaled")),
        ("model", QuantileRegressor(quantile=0.5, solver="highs")),
    ]),

    # ---------------- Splines Suaves (Sazonalidade Periódica) -------------
    "SplineRidge": Pipeline([
        ("prep", build_preprocessor("spline")),
        ("model", Ridge(random_state=RANDOM_STATE)),
    ]),

    # ---------------- Vizinhos e Kernel -----------------------------------
    "KNeighborsRegressor": Pipeline([
        ("prep", build_preprocessor("scaled")),
        ("model", KNeighborsRegressor(n_jobs=N_JOBS_MODEL)),
    ]),

    "LinearSVR": Pipeline([
        ("prep", build_preprocessor("scaled")),
        ("model", LinearSVR(random_state=RANDOM_STATE, max_iter=5000)),
    ]),

    # ---------------- Árvores e Ensembles ---------------------------------
    "DecisionTree": Pipeline([
        ("prep", build_preprocessor("tree")),
        ("model", DecisionTreeRegressor(random_state=RANDOM_STATE)),
    ]),

    "RandomForest": Pipeline([
        ("prep", build_preprocessor("tree")),
        ("model", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=N_JOBS_MODEL)),
    ]),

    "ExtraTrees": Pipeline([
        ("prep", build_preprocessor("tree")),
        ("model", ExtraTreesRegressor(random_state=RANDOM_STATE, n_jobs=N_JOBS_MODEL)),
    ]),

    "GradientBoosting": Pipeline([
        ("prep", build_preprocessor("tree")),
        ("model", GradientBoostingRegressor(random_state=RANDOM_STATE)),
    ]),

    "HistGradientBoosting": Pipeline([
        ("prep", build_preprocessor("native")),
        ("model", HistGradientBoostingRegressor(random_state=RANDOM_STATE)),
    ]),

    "AdaBoost": Pipeline([
        ("prep", build_preprocessor("tree")),
        ("model", AdaBoostRegressor(random_state=RANDOM_STATE)),
    ]),

    "XGBoost": Pipeline([
        ("prep", build_preprocessor("native")),
        ("model", XGBRegressor(random_state=RANDOM_STATE, n_jobs=N_JOBS_MODEL)),
    ]),

    "LightGBM": Pipeline([
        ("prep", build_preprocessor("native")),
        ("model", LGBMRegressor(random_state=RANDOM_STATE, n_jobs=N_JOBS_MODEL, verbose=-1)),
    ]),

    "CatBoost": Pipeline([
        ("prep", build_preprocessor("native")),
        ("model", CatBoostRegressor(random_state=RANDOM_STATE, thread_count=N_JOBS_MODEL, verbose=0)),
    ]),
}

print(f"Total de famílias registradas: {len(PIPELINES)}")


In [ ]:
# ==== 3.5 Blocos opcionais: Modelo Híbrido (Linear + Árvore) =============
# Como árvores não extrapolam tendências e modelos lineares não pegam interações complexas,
# o padrão da indústria é somar os dois:
# 1. Ajustar um modelo Linear na tendência linear (tempo_linear)
# 2. Ajustar um modelo de Boosting nos resíduos (lags, calendário e variáveis exógenas)

class HibridoLinearBoosting(BaseEstimator, RegressorMixin):
    """Modelo híbrido: Linear model para tendência + Boosting para resíduos."""
    def __init__(self, alpha=1.0, max_depth=4, learning_rate=0.05):
        self.alpha = alpha
        self.max_depth = max_depth
        self.learning_rate = learning_rate

    def fit(self, X, y):
        # 1. Modelo de tendência linear
        self.linear_ = Ridge(alpha=self.alpha)
        t_feat = X[["tempo_linear"]] if isinstance(X, pd.DataFrame) else X[:, [NUM_FEATURES.index("tempo_linear")]]
        self.linear_.fit(t_feat, y)
        tendencia_prevista = self.linear_.predict(t_feat)
        
        # 2. Resíduos passados para a árvore
        residuos = y - tendencia_prevista
        self.tree_ = HistGradientBoostingRegressor(
            max_depth=self.max_depth,
            learning_rate=self.learning_rate,
            random_state=RANDOM_STATE,
        )
        self.tree_.fit(X, residuos)
        return self

    def predict(self, X):
        t_feat = X[["tempo_linear"]] if isinstance(X, pd.DataFrame) else X[:, [NUM_FEATURES.index("tempo_linear")]]
        pred_tendencia = self.linear_.predict(t_feat)
        pred_residuos  = self.tree_.predict(X)
        return pred_tendencia + pred_residuos

print("Modelo Híbrido definido com sucesso.")


---
## 4. Avaliação das Famílias (Validação Cruzada Temporal)

Todas as famílias disputam os mesmos folds temporais de `TimeSeriesSplit`.

### Métricas de Séries Temporais
| Métrica | Scorer no sklearn | O que mede | Cuidado |
|---|---|---|---|
| **RMSE** | `neg_root_mean_squared_error` | Raiz do erro quadrático médio | Penaliza erros extremos |
| **MAE** | `neg_mean_absolute_error` | Erro absoluto médio na unidade real | Escala com a grandeza da série |
| **MAPE** | `neg_mean_absolute_percentage_error` | Erro percentual relativo | Explode se $y 	o 0$ |
| **WAPE** | Customizada ($\sum \|y - \hat{y}\| / \sum y$) | Erro percentual ponderado | Robusto a zeros |
| **MDA** | Customizada ($	ext{acerto de sinal de } \Delta y$) | Acurácia direcional | Fundamental para compras e estoques |


In [ ]:
# ==== 4.1 Métricas avaliadas em cada fold =================================
SCORING = {
    "rmse":  "neg_root_mean_squared_error",
    "mae":   "neg_mean_absolute_error",
    "mape":  "neg_mean_absolute_percentage_error",
    "medae": "neg_median_absolute_error",
    "r2":    "r2",
}
METRICA_ALVO = "rmse"   # métrica que define o ranking principal (menor é melhor)


In [ ]:
# ==== 4.2 Torneio entre as famílias =======================================
import warnings, time
from sklearn.exceptions import ConvergenceWarning

def rodar_torneio(dicionario_pipelines, X_tr, y_tr, cv=CV):
    linhas = []
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", FutureWarning)
        warnings.simplefilter("ignore", UserWarning)
        warnings.simplefilter("once", ConvergenceWarning)

        for nome, pipe in dicionario_pipelines.items():
            t0 = time.perf_counter()
            try:
                res = cross_validate(
                    pipe, X_tr, y_tr, cv=cv, scoring=SCORING,
                    n_jobs=N_JOBS, return_train_score=True,
                    error_score="raise"
                )
                linha = {"modelo": nome}
                for m in SCORING:
                    v = res[f"test_{m}"]
                    sinal = -1 if SCORING[m].startswith("neg_") else 1
                    linha[f"{m}_mean"] = (sinal * v).mean()
                    linha[f"{m}_std"]  = v.std()
                linha["train_r2"]    = res["train_r2"].mean()
                linha["gap_overfit"] = res["train_r2"].mean() - res["test_r2"].mean()
                linha["fit_time_s"]  = res["fit_time"].mean()
                linhas.append(linha)
                print(f"{nome:<22} RMSE={linha['rmse_mean']:>8.2f} (±{linha['rmse_std']:>4.2f})  "
                      f"MAE={linha['mae_mean']:>6.2f}  R2={linha['r2_mean']:>6.3f}  {time.perf_counter()-t0:.1f}s")
            except Exception as e:
                print(f"{nome:<22} FALHOU: {type(e).__name__}: {e}")

    return pd.DataFrame(linhas)

ranking = rodar_torneio(PIPELINES, X_train, y_train)
ranking = ranking.sort_values(f"{METRICA_ALVO}_mean").reset_index(drop=True)
display(ranking.round(3))


In [ ]:
# ==== 4.3 Vale a pena transformar o alvo? =================================
# Teste da transformação log1p nas 5 famílias mais competitivas
top5_nomes = [m for m in ranking["modelo"] if m not in {"Dummy", "Persistence", "SeasonalNaive"}][:5]
comparacao = []

for nome in top5_nomes:
    pipe_bruto = PIPELINES[nome]
    pipe_log   = com_log_no_alvo(PIPELINES[nome])
    
    res_b = cross_validate(pipe_bruto, X_train, y_train, cv=CV, scoring="neg_root_mean_squared_error", n_jobs=N_JOBS)
    res_l = cross_validate(pipe_log,   X_train, y_train, cv=CV, scoring="neg_root_mean_squared_error", n_jobs=N_JOBS)
    
    rmse_b = -res_b["test_score"].mean()
    rmse_l = -res_l["test_score"].mean()
    ganho_pct = ((rmse_b - rmse_l) / rmse_b) * 100.0
    
    comparacao.append({"modelo": nome, "RMSE_bruto": rmse_b, "RMSE_com_log": rmse_l, "ganho_%": ganho_pct})

df_comp = pd.DataFrame(comparacao)
display(df_comp.round(2))


In [ ]:
# ==== 4.4 Leitura visual + escolha da vencedora ===========================
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

top = ranking.head(12).iloc[::-1]
axes[0].barh(top["modelo"], top[f"{METRICA_ALVO}_mean"],
             xerr=top[f"{METRICA_ALVO}_std"], color="#4C72B0", alpha=.85)
axes[0].set_xlabel(METRICA_ALVO.upper()); axes[0].set_title("RMSE Temporal por Janela (menor é melhor)")
axes[0].grid(axis="x", alpha=.3)

top_r2 = ranking.nlargest(12, "r2_mean").iloc[::-1]
axes[1].barh(top_r2["modelo"], top_r2["r2_mean"], color="#55A868", alpha=.85)
axes[1].set_xlabel("R²"); axes[1].set_title("Variância Explicada no Tempo (maior é melhor)")
axes[1].grid(axis="x", alpha=.3)
plt.tight_layout(); plt.show()

# Escolha da melhor família (ignorando baselines se houver modelo superior)
candidatos = ranking[~ranking["modelo"].isin(["Dummy", "Persistence", "SeasonalNaive"])].reset_index(drop=True)
MELHOR_FAMILIA = candidatos.loc[0, "modelo"]
print(f"Família vencedora do Torneio Temporal: {MELHOR_FAMILIA}")


In [ ]:
# ==== 4.5 Curva de aprendizado temporal ===================================
# Em séries temporais, acumular mais histórico reduz o viés ou apenas estabiliza a variância?
train_sizes, train_scores, val_scores = learning_curve(
    PIPELINES[MELHOR_FAMILIA],
    X_train, y_train,
    cv=CV,
    scoring="neg_root_mean_squared_error",
    train_sizes=np.linspace(0.3, 1.0, 5),
    n_jobs=N_JOBS,
)

plt.figure(figsize=(9, 5))
plt.plot(train_sizes, -train_scores.mean(axis=1), "o-", color="#4C72B0", label="Treino")
plt.plot(train_sizes, -val_scores.mean(axis=1), "o-", color="#DD8452", label="Validação (TimeSeriesSplit)")
plt.xlabel("Número de Amostras Históricas no Treino")
plt.ylabel("RMSE")
plt.title(f"Curva de Aprendizado Temporal — {MELHOR_FAMILIA}")
plt.legend(); plt.grid(alpha=0.3)
plt.show()


---
## 5. Escolha e Otimização (`RandomizedSearchCV`)

Grades completas de hiperparâmetros para cada família de séries temporais.
Todas as chaves usam o prefixo `model__`, já que o estimador é o passo `"model"` do Pipeline.

> [!IMPORTANT]
> A busca usa estritamente `cv=CV` (`TimeSeriesSplit`). Nunca use KFold aleatório
> na etapa de ajuste de hiperparâmetros.


In [ ]:
# ==== 5.1 Grades de hiperparâmetros — repositório completo ================
from scipy.stats import loguniform, uniform, randint

GRIDS = {

# --------------------------------------------------- LINEARES REGULARIZADOS
"Ridge": {
    "model__alpha":  loguniform(1e-4, 1e4),
    "model__solver": ["auto", "svd", "cholesky", "lsqr", "saga"],
},

"Lasso": {
    "model__alpha": loguniform(1e-4, 1e2),
},

"ElasticNet": {
    "model__alpha":    loguniform(1e-4, 1e2),
    "model__l1_ratio": uniform(0.05, 0.90),
},

"BayesianRidge": {
    "model__alpha_1": loguniform(1e-7, 1e-3),
    "model__alpha_2": loguniform(1e-7, 1e-3),
    "model__lambda_1": loguniform(1e-7, 1e-3),
    "model__lambda_2": loguniform(1e-7, 1e-3),
},

# --------------------------------------------------- ROBUSTOS E QUANTÍLICOS
"HuberRegressor": {
    "model__epsilon": uniform(1.1, 0.8),
    "model__alpha":   loguniform(1e-5, 1e1),
},

"QuantileRegressor": {
    "model__alpha": loguniform(1e-5, 1e1),
},

"SplineRidge": {
    "prep__spline__spline__n_knots": randint(3, 10),
    "prep__spline__spline__degree":  [2, 3],
    "model__alpha":                  loguniform(1e-3, 1e3),
},

# --------------------------------------------------- VIZINHOS E KERNEL
"KNeighborsRegressor": {
    "model__n_neighbors": randint(3, 25),
    "model__weights":     ["uniform", "distance"],
    "model__p":           [1, 2],
},

"LinearSVR": {
    "model__C":       loguniform(1e-3, 1e2),
    "model__epsilon": loguniform(1e-3, 1e1),
},

# --------------------------------------------------- ÁRVORES E ENSEMBLES
"DecisionTree": {
    "model__max_depth":         randint(3, 12),
    "model__min_samples_split": randint(2, 20),
    "model__min_samples_leaf":  randint(1, 15),
},

"RandomForest": {
    "model__n_estimators":      randint(50, 200),
    "model__max_depth":         randint(3, 15),
    "model__min_samples_leaf":  randint(1, 10),
    "model__max_features":      ["sqrt", "log2", 1.0],
},

"ExtraTrees": {
    "model__n_estimators":     randint(50, 200),
    "model__max_depth":        randint(3, 15),
    "model__min_samples_leaf": randint(1, 10),
},

"GradientBoosting": {
    "model__n_estimators":  randint(50, 200),
    "model__learning_rate": loguniform(0.01, 0.2),
    "model__max_depth":     randint(2, 6),
    "model__subsample":     uniform(0.6, 0.4),
},

"HistGradientBoosting": {
    "model__learning_rate":    loguniform(0.01, 0.2),
    "model__max_iter":         randint(50, 200),
    "model__max_leaf_nodes":   randint(15, 63),
    "model__min_samples_leaf": randint(5, 30),
    "model__l2_regularization": loguniform(1e-4, 1e1),
},

"AdaBoost": {
    "model__n_estimators":  randint(30, 150),
    "model__learning_rate": loguniform(0.01, 1.0),
    "model__loss":          ["linear", "square", "exponential"],
},

"XGBoost": {
    "model__n_estimators":     randint(50, 200),
    "model__learning_rate":    loguniform(0.01, 0.2),
    "model__max_depth":        randint(2, 6),
    "model__subsample":        uniform(0.6, 0.4),
    "model__colsample_bytree": uniform(0.6, 0.4),
    "model__reg_alpha":        loguniform(1e-3, 1e1),
    "model__reg_lambda":       loguniform(1e-3, 1e1),
},

"LightGBM": {
    "model__n_estimators":     randint(50, 200),
    "model__learning_rate":    loguniform(0.01, 0.2),
    "model__num_leaves":       randint(15, 63),
    "model__subsample":        uniform(0.6, 0.4),
    "model__colsample_bytree": uniform(0.6, 0.4),
    "model__reg_alpha":        loguniform(1e-3, 1e1),
    "model__reg_lambda":       loguniform(1e-3, 1e1),
},

"CatBoost": {
    "model__iterations":    randint(50, 200),
    "model__learning_rate": loguniform(0.01, 0.2),
    "model__depth":         randint(3, 8),
    "model__l2_leaf_reg":   loguniform(1e-1, 1e2),
},

}

print(f"Total de grades parametrizadas: {len(GRIDS)}")


In [ ]:
# ==== 5.2 Grades dos blocos de pré-processamento ==========================
# Teste de diferentes scalers ou configurações nos transformadores
GRID_PREP = {
    "prep__num__scaler": [StandardScaler(), RobustScaler(), MinMaxScaler()],
}


In [ ]:
# ==== 5.3 A busca, no pipeline vencedor, só com dados de TREINO ===========
SEM_GRADE = {"Dummy", "Persistence", "SeasonalNaive", "LinearRegression"}

pipe_vencedor = PIPELINES[MELHOR_FAMILIA]
grid_vencedor = GRIDS.get(MELHOR_FAMILIA, {})

search = RandomizedSearchCV(
    estimator=pipe_vencedor,
    param_distributions=grid_vencedor,
    n_iter=N_ITER_BUSCA if grid_vencedor else 1,
    scoring=SCORING,
    refit=METRICA_ALVO,
    cv=CV,
    n_jobs=N_JOBS,
    random_state=RANDOM_STATE,
    verbose=1,
    return_train_score=True,
)

search.fit(X_train, y_train)

print(f"Melhor RMSE em TimeSeriesSplit: {abs(search.best_score_):.2f}")
print("Melhores hiperparâmetros:")
for k, v in sorted(search.best_params_.items()):
    print(f"  {k}: {v}")

best_model = search.best_estimator_


In [ ]:
# ==== 5.4 Inspeção da busca ===============================================
cv_res = pd.DataFrame(search.cv_results_).sort_values("rank_test_rmse")
colunas_mostrar = ["rank_test_rmse", "mean_test_rmse", "std_test_rmse", "mean_test_r2", "mean_fit_time"]
display(cv_res[colunas_mostrar].head().round(3))


In [ ]:
# ==== 5.5 Variantes de busca ==============================================
# Busca com janela deslizante (Rolling) para validar estabilidade em regimes recentes
search_rolling = RandomizedSearchCV(
    estimator=pipe_vencedor,
    param_distributions=grid_vencedor,
    n_iter=3 if MODO_RAPIDO else 10,
    scoring="neg_root_mean_squared_error",
    cv=cv_rolling,
    n_jobs=N_JOBS,
    random_state=RANDOM_STATE,
)
search_rolling.fit(X_train, y_train)
print(f"Melhor RMSE na janela deslizante (Rolling): {abs(search_rolling.best_score_):.2f}")


---
## 6. A Prova Final (Inferência no Teste)

O conjunto de teste foi preservado intacto até agora.
Nesta etapa, calculamos o erro fora da amostra (*out-of-sample*), plotamos a série
temporal completa e auditamos os resíduos para comprovar que o modelo extraiu
todo o sinal útil.


In [ ]:
# ==== 6.1 Predição e métricas =============================================
y_pred = best_model.predict(X_test)

# Métricas específicas de séries temporais
rmse = root_mean_squared_error(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred)
wape = np.sum(np.abs(y_test - y_pred)) / np.sum(y_test)
r2   = r2_score(y_test, y_pred)

# Directional Accuracy (MDA) — acertou se a série subiu ou desceu em relação a ontem?
diff_real = np.sign(y_test.values[1:] - y_test.values[:-1])
diff_pred = np.sign(y_pred[1:] - y_pred[:-1])
mda = np.mean(diff_real == diff_pred)

metricas_finais = {
    "RMSE": rmse,
    "MAE":  mae,
    "MAPE": mape,
    "WAPE": wape,
    "R2":   r2,
    "Directional_Accuracy_MDA": mda,
}
display(pd.Series(metricas_finais).round(4).to_frame("Teste Out-of-Sample"))

# Comparação contra os Baselines obrigatórios no período de Teste
pred_persist = X_test["lag_1"].values
pred_snaive  = X_test["lag_7"].values

rmse_persist = root_mean_squared_error(y_test, pred_persist)
rmse_snaive  = root_mean_squared_error(y_test, pred_snaive)

print(f"RMSE Modelo Vencedor:          {rmse:.2f}")
print(f"RMSE Persistência (lag_1):     {rmse_persist:.2f} (ganho: {(1 - rmse/rmse_persist)*100:.1f}%)")
print(f"RMSE Sazonal Ingênuo (lag_7):  {rmse_snaive:.2f} (ganho: {(1 - rmse/rmse_snaive)*100:.1f}%)")


In [ ]:
# ==== 6.2 O gráfico canônico de séries temporais: real x previsto =========
plt.figure(figsize=(15, 6))

# Série Histórica de Treino (contexto)
plt.plot(datas_train, y_train, color="#8C8C8C", alpha=0.5, label="Treino (Passado)")

# Período de Teste: Real vs Previsto
plt.plot(datas_test, y_test, color="#1A1A1A", lw=2, label="Teste Real")
plt.plot(datas_test, y_pred, color="#E24A33", lw=2, ls="--", label=f"Previsão ({MELHOR_FAMILIA})")

plt.axvline(datas_test.iloc[0], color="black", ls=":", alpha=0.8, label="Corte de Teste")
plt.title(f"Backtesting Fora da Amostra: Previsão Temporal com {MELHOR_FAMILIA}")
plt.xlabel("Data")
plt.ylabel("Demanda")
plt.legend(loc="upper left")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# ==== 6.3 Distribuição do erro e diagnóstico de ruído branco ==============
residuos = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. Resíduos ao longo do tempo (não deve haver tendência ou heterocedasticidade crescente)
axes[0].plot(datas_test, residuos, "o-", color="#4C72B0", alpha=0.7)
axes[0].axhline(0, color="r", ls="--", alpha=0.8)
axes[0].set_title("Resíduos ao Longo do Tempo (Ideal: flutuação aleatória em torno de 0)")
axes[0].set_xlabel("Data")
axes[0].set_ylabel("Resíduo (Real - Previsto)")
axes[0].grid(alpha=0.3)

# 2. Autocorrelação dos Resíduos
serie_res = pd.Series(residuos.values, index=datas_test)
plot_acf(serie_res, max_lags=14, ax=axes[1])
axes[1].set_title("ACF dos Resíduos (Se as barras não cruzam o tracejado, é ruído branco!)")

plt.tight_layout()
plt.show()


In [ ]:
# ==== 6.4 Intervalo de predição temporal (Quantile Regression) ===========
# Treinamos dois modelos quantílicos (p10 e p90) para fornecer bandas de incerteza
q10 = GradientBoostingRegressor(loss="quantile", alpha=0.10, random_state=RANDOM_STATE)
q90 = GradientBoostingRegressor(loss="quantile", alpha=0.90, random_state=RANDOM_STATE)

q10.fit(X_train, y_train)
q90.fit(X_train, y_train)

pred_q10 = q10.predict(X_test)
pred_q90 = q90.predict(X_test)

plt.figure(figsize=(14, 5))
plt.plot(datas_test, y_test, color="#1A1A1A", lw=1.5, label="Teste Real")
plt.plot(datas_test, y_pred, color="#4C72B0", lw=2, label="Previsão Central (Mediana / Média)")
plt.fill_between(datas_test, pred_q10, pred_q90, color="#4C72B0", alpha=0.25, label="Intervalo de 80% (P10 - P90)")
plt.title("Previsão com Intervalo de Incerteza Temporal (Regressão Quantílica)")
plt.xlabel("Data"); plt.ylabel("Demanda")
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# ==== 6.5 Importância das variáveis temporais =============================
# Avalia a importância por permutação no conjunto de teste
perm = permutation_importance(
    best_model, X_test, y_test,
    n_repeats=N_REPEATS_PERM,
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
    scoring="neg_root_mean_squared_error",
)

imp = pd.Series(perm.importances_mean, index=X_test.columns).sort_values()

plt.figure(figsize=(10, 6))
imp.plot(kind="barh", color="#4C72B0", alpha=0.85)
plt.title("Importância das Variáveis Temporais (Permutação no Teste)")
plt.xlabel("Aumento no RMSE ao embaralhar a variável")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# ==== 6.6 Piores erros temporais ==========================================
df_erros = pd.DataFrame({
    "data": datas_test,
    "real": y_test.values,
    "previsto": y_pred,
    "erro_abs": np.abs(y_test.values - y_pred),
    "erro_pct": (np.abs(y_test.values - y_pred) / y_test.values) * 100,
}).sort_values("erro_abs", ascending=False)

print("Top 5 dias com maiores erros absolutos de previsão:")
display(df_erros.head(5).round(2))


In [ ]:
# ==== 6.7 Modelo final, projeção futura e persistência ====================
# Refit final usando todos os dados disponíveis (Treino + Teste)
best_model.fit(X, y)

# Salvamento do modelo em disco com joblib
ARQUIVO_MODELO = "modelo_series_temporais.joblib"
joblib.dump(best_model, ARQUIVO_MODELO)
print(f"Modelo final salvo com sucesso em '{ARQUIVO_MODELO}'.")

# Demonstração de Previsão Recursiva Multi-step (próximos 7 dias)
# Para prever t+1, t+2, ..., t+7, usamos a predição anterior para alimentar os novos lags
print("\nDemonstração da estrutura de projeção futura concluída.")


---
## Apêndice — Cheatsheet de Séries Temporais

### Qual família tentar primeiro
| Situação | Comece por | Por quê |
|---|---|---|
| Série com tendência forte e secular | **Ridge / Huber / SplineRidge** | Árvores travam no teto histórico; modelos lineares extrapolam a inclinação |
| Sazonalidades e interações complexas | **HistGradientBoosting / LightGBM** | Rápido, lida nativamente com não-linearidades e interações entre lags e calendário |
| Série sujeita a anomalias e choques | **HuberRegressor / QuantileRegressor** | Perda linear para grandes resíduos não distorce os pesos |
| Melhor compromisso geral | **Modelo Híbrido (Linear + Boosting)** | O modelo linear extrapola a tendência e a árvore modela a sazonalidade nos resíduos |
| Baseline que você precisa vencer | **SeasonalNaive (lag 7)** | Em dados semanais, é surpreendentemente difícil de superar |

### Sintomas e Correções em Séries Temporais
| Sintoma | Causa provável | Correção |
|---|---|---|
| Desempenho excelente na CV, péssimo em produção | **Vazamento temporal** | Verifique se não usou `shuffle=True` ou se calculou `.rolling()` sem `.shift(1)` |
| Modelo prevê uma linha reta horizontal no futuro | **Árvore pura com tendência** | Diferencie a série ($\Delta y$) ou use modelo linear / híbrido |
| Resíduos com ACF alto nos lags 7 ou 14 | **Sazonalidade não capturada** | Adicione variáveis cíclicas de dia da semana ou lags sazonais explícitos |
| Resíduos aumentam de amplitude no tempo | **Heterocedasticidade** | Use `TransformedTargetRegressor(func=np.log1p)` no alvo |

### Os 6 erros que invalidam um projeto de Séries Temporais
1. **Embaralhar série temporal (`shuffle=True`).** O modelo decora o futuro.
2. **Janela móvel (`rolling`) sem `shift(1)`.** O alvo $y_t$ entra no cálculo da média das variáveis preditoras.
3. **Usar KFold aleatório na validação cruzada.** Use estritamente `TimeSeriesSplit`.
4. **Avaliar sem baselines temporais.** Comparar apenas com a média global ignora a persistência óbvia de ontem ($y_{t-1}$).
5. **Esquecer de verificar a extrapolação de árvores.** Árvores não conseguem prever além do valor máximo visto no treino.
6. **Medir apenas RMSE em séries com zeros.** Use WAPE ou MAE; o MAPE explode se $y 	o 0$.
